# 04 · Anatomy: registration and the Allen atlas

Following the tutorial notebook [`04_anatomy_allen_student.ipynb`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/blob/main/notebooks/level2/04_anatomy_allen_student.ipynb).

**Kernel:** `cajal-lipidomics`

**What I'm doing here**
- Affine + nonlinear warp, what ABBA does, CCF coordinate -> voxel -> region
- Region maps, contours, slicing angle (pregnant section is tilted)
- Grey vs white matter from the Allen color #cccccc, HexCer 42:2 as the myelin check
- Region x lipid mean matrix + sorted heatmap

**My notes / what surprised me**
- ...

In [ ]:
# ---- setup: the same first cell in every notebook ---------------------------------------

# My own little package. `paths` knows where the tutorial clone, its provided data,
# and my own outputs live, so I never type "../../data/..." by hand.
from lipid_project import paths

paths.ensure_dirs()   # creates data/derived/, figures/, results/ if they're missing (safe to re-run)
paths.summary()       # prints where everything points + which stages I've built so far.
                      # If a line starts with "--", that thing is missing: fix it before going on.

# The usual scientific stack
import numpy as np                 # arrays and math
import pandas as pd                # tables (obs / var are pandas DataFrames)
import matplotlib.pyplot as plt    # plotting
import anndata as ad               # the AnnData container: X (pixels x lipids) + obs + var + obsm + layers

# The tutorial's helper package (installed from my clone in external/).
# Rule from the course: before calling any cl.<module>.<function>, open
# external/lipidomics_tutorial_cajalcourse/src/cajal_lipidomics/<module>.py and read it.
import cajal_lipidomics as cl
cl.style.set_style()               # the course's figure defaults, so plots look clean and consistent

# If this prints without a red error, the kernel is right. A ModuleNotFoundError here almost
# always means the wrong kernel: re-pick "cajal-lipidomics" in the top-right kernel picker.
print("ready. cajal_lipidomics", cl.__version__)

## load

In [ ]:
# ---- load the previous stage --------------------------------------------------------------
# My NB03 output. Everything registration produced is already sitting in adata.obs:
# xccf / yccf / zccf (Allen coordinates in mm), acronym (region), allencolor (region color).
adata = ad.read_h5ad(paths.stage("normalized"))

# X still holds the RAW intensities. For anatomy + lipid views I want the uMAIA-normalized
# values, so I swap them in once, right here, and everything below uses them.
# (This only changes the object in memory. The file on disk keeps raw X + the umaia layer.)
adata.X = adata.layers["umaia"]

print("pixels x lipids:", adata.shape)
print("Allen regions on these sections:", adata.obs["acronym"].nunique())   # expect ~174

# No output file: this notebook explores anatomy and builds the region x lipid view,
# but doesn't create a new pipeline stage. NB05 picks up from 03_normalized again.

## my work

In [ ]:
# my code from here on, one small step per cell, and look at the output each time